In [ ]:
"""
sandbox_time.ipynb

A sandbox to develop a time-resolved class.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
from core.data import default_sessions

subj_id = "MR82"
sess_id = default_sessions[subj_id]

In [ ]:
from sg.models import StrategyEncoder, make_tre

encoder_mb = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mb",
    strategy_params="mb",
    add_interaction=True,
    add_svd=True,
    num_svd=10,
)
encoder_mb.verify()

encoder_mf = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mf",
    strategy_params="mf",
    add_interaction=True,
    add_svd=True,
    num_svd=10,
)
encoder_mf.verify()

In [ ]:
import numpy as np

pivots_groups_mb = {
    regr: [f"{regr}_{i}" for i in range(encoder_mb.num_bins)]
    for regr in np.concatenate((encoder_mb.tv_keys, encoder_mb.tv_names["interaction"]))
}
pivots_groups_mb["spe"] = [f"spe_{i}" for i in range(encoder_mb.num_bins)]
pivots_groups_mb["svd"] = [f"svd_{i}" for i in range(encoder_mb.num_svd)]

pivots_groups_mf = {
    regr: [f"{regr}_{i}" for i in range(encoder_mf.num_bins)]
    for regr in np.concatenate((encoder_mf.tv_keys, encoder_mf.tv_names["interaction"]))
}
pivots_groups_mf["rpe"] = [f"rpe_{i}" for i in range(encoder_mf.num_bins)]
pivots_groups_mf["svd"] = [f"svd_{i}" for i in range(encoder_mf.num_svd)]

In [ ]:
cvr2s = {
    strat: {
        regr: {reg: [] for reg in encoder_mb.regions} for regr in pivots_groups_.keys()
    }
    for strat, pivots_groups_ in {
        "mb": pivots_groups_mb,
        "mf": pivots_groups_mf,
    }.items()
}
dr2s = {
    strat: {
        regr: {reg: [] for reg in encoder_mb.regions} for regr in pivots_groups_.keys()
    }
    for strat, pivots_groups_ in {
        "mb": pivots_groups_mb,
        "mf": pivots_groups_mf,
    }.items()
}

In [ ]:
import pickle
from sg.models import ShuffledEncoder
from core.data import subject_ids, session_ids
from utils.paths import VARS_DIR

for sess_id in session_ids[np.where(subject_ids == subj_id)[0][0]]:
    print(f"> {sess_id}")
    try:
        se_mb = ShuffledEncoder(
            subj_id,
            sess_id,
            enc_class=make_tre(StrategyEncoder, tr_type="dme"),
            strategy_filter="mb",
            strategy_params="mb",
            add_interaction=True,
            norm=True,
            add_svd=True,
            num_svd=10,
            stepsize_s=0.25,
        )

        se_mf = ShuffledEncoder(
            subj_id,
            sess_id,
            enc_class=make_tre(StrategyEncoder, tr_type="dme"),
            strategy_filter="mf",
            strategy_params="mf",
            add_interaction=True,
            norm=True,
            add_svd=True,
            num_svd=10,
            stepsize_s=0.25,
        )

        se_mb.get_cvr2_groups(pivots_groups=pivots_groups_mb, n_iters=5)
        se_mb.get_dr2_groups(pivots_groups=pivots_groups_mb, n_iters=5)

        se_mf.get_cvr2_groups(pivots_groups=pivots_groups_mf, n_iters=5)
        se_mf.get_dr2_groups(pivots_groups=pivots_groups_mf, n_iters=5)
    except ValueError as e:
        print(f"{subj_id} {sess_id} skipped: {e}")
        continue

    assert all(se_mb.encoder_full.regions == se_mf.encoder_full.regions), "ruh roh"

    for reg in se_mb.encoder_full.regions:
        for strat, (se, pivots_groups_) in {
            "mb": (se_mb, pivots_groups_mb),
            "mf": (se_mf, pivots_groups_mf),
        }.items():
            for regr in pivots_groups_.keys():
                cvr2s[strat][regr][reg].append(
                    se.cvr2_unit[regr].mean(axis=0)[se.encoder_full.reg_idxs[reg]]
                )
                dr2s[strat][regr][reg].append(
                    se.dr2_unit[regr].mean(axis=0)[se.encoder_full.reg_idxs[reg]]
                )

    fpath = VARS_DIR / "cv_d_r2" / "grouped"
    fpath.mkdir(exist_ok=True)
    with open(fpath / f"{subj_id}_cvr2s.pkl", "wb") as f:
        pickle.dump(cvr2s, f)
    with open(fpath / f"{subj_id}_dr2s.pkl", "wb") as f:
        pickle.dump(dr2s, f)

In [ ]:
from core.viz import plot_grouped_boxes
from utils.colors import colors_region
from utils.viz_utils import save_fig
from utils.paths import FIGURES_DIR

fig, axes = plt.subplots(
    ncols=2, nrows=2, figsize=(5, 5), sharey=True, tight_layout=True
)

for i, tv_type in enumerate(["base_regressors", "interaction"]):
    for j, (strategy, (metrics, regressors_)) in enumerate(
        {
            "mb": (cvr2s["mb"], encoder_mb.tv_names[tv_type]),
            "mf": (cvr2s["mf"], encoder_mf.tv_names[tv_type]),
        }.items()
    ):
        _, ax = plot_grouped_boxes(
            data={
                group: {
                    reg: np.concatenate(metrics[group][reg])
                    for reg in encoder_mb.regions
                }
                for group in regressors_
            },
            colors=colors_region,
            ax=axes[i][j],
        )
        ax.legend(loc="upper right")
        ax.set_title(strategy)

fig.suptitle(rf"{subj_id}, cv $r^2$")
save_fig(
    fig,
    FIGURES_DIR / "cv_d_r2" / "boxplot_comp" / "grouped",
    f"{subj_id}_cvr2_comp.png",
)

fig, axes = plt.subplots(
    ncols=2, nrows=2, figsize=(5, 5), sharey=True, tight_layout=True
)
for i, tv_type in enumerate(["base_regressors", "interaction"]):
    for j, (strategy, (metrics, regressors_)) in enumerate(
        {
            "mb": (dr2s["mb"], encoder_mb.tv_names[tv_type]),
            "mf": (dr2s["mf"], encoder_mf.tv_names[tv_type]),
        }.items()
    ):
        _, ax = plot_grouped_boxes(
            data={
                group: {
                    reg: np.concatenate(metrics[group][reg])
                    for reg in encoder_mb.regions
                }
                for group in regressors_
            },
            colors=colors_region,
            ax=axes[i][j],
        )
        ax.legend(loc="upper right")
        ax.set_title(strategy)

fig.suptitle(rf"{subj_id}, $\Delta r^2$")
save_fig(
    fig,
    FIGURES_DIR / "cv_d_r2" / "boxplot_comp" / "grouped",
    f"{subj_id}_dr2_comp.png",
)

## replot boxplots

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sg.models import make_tre, StrategyEncoder
from core.data import default_sessions
from core.viz import plot_grouped_boxes
from utils.colors import colors_region
from utils.paths import VARS_DIR, FIGURES_DIR
from utils.viz_utils import save_fig

subj_id = "MR94"

encoder_mb = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    default_sessions[subj_id],
    norm=True,
    stepsize_s=0.5,
    strategy_filter="mb",
    strategy_params="mb",
    add_interaction=True,
    # add_svd=True,
    # num_svd=10,
)
encoder_mb.verify()

encoder_mf = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    default_sessions[subj_id],
    norm=True,
    stepsize_s=0.5,
    strategy_filter="mf",
    strategy_params="mf",
    add_interaction=True,
    # add_svd=True,
    # num_svd=10,
)
encoder_mf.verify()

fpath = VARS_DIR / "cv_d_r2" / "grouped" / "no_svd"
with open(fpath / f"{subj_id}_cvr2s.pkl", "rb") as f:
    cvr2s = pickle.load(f)
with open(fpath / f"{subj_id}_dr2s.pkl", "rb") as f:
    dr2s = pickle.load(f)

In [ ]:
for plot_points in [True, False]:
    points_str = "points" if plot_points else "clean"

    fig, axes = plt.subplots(
        ncols=2, nrows=2, figsize=(6, 6), sharey=True, tight_layout=True
    )

    for i, tv_type in enumerate(["base_regressors", "interaction"]):
        for j, (strategy, (metrics, regressors_)) in enumerate(
            {
                "mb": (cvr2s["mb"], encoder_mb.tv_names[tv_type]),
                "mf": (cvr2s["mf"], encoder_mf.tv_names[tv_type]),
            }.items()
        ):
            _, ax = plot_grouped_boxes(
                data={
                    group: {
                        reg: np.concatenate(metrics[group][reg])
                        for reg in encoder_mb.regions
                    }
                    for group in regressors_
                },
                plot_points=plot_points,
                colors=colors_region,
                ax=axes[i][j],
            )
            ax.legend(loc="upper right")
            ax.set_title(strategy)
            if plot_points:
                ax.set_ylim([-0.3, 1.1])
            else:
                ax.set_ylim([-0.05, 0.5])

    fig.suptitle(rf"{subj_id}, cv $r^2$")
    save_fig(
        fig,
        FIGURES_DIR / "cv_d_r2" / "boxplot_comp" / "grouped" / "no_svd",
        f"{subj_id}_cvr2_comp-{points_str}.png",
    )

    fig, axes = plt.subplots(
        ncols=2, nrows=2, figsize=(6, 6), sharey=True, tight_layout=True
    )
    for i, tv_type in enumerate(["base_regressors", "interaction"]):
        for j, (strategy, (metrics, regressors_)) in enumerate(
            {
                "mb": (dr2s["mb"], encoder_mb.tv_names[tv_type]),
                "mf": (dr2s["mf"], encoder_mf.tv_names[tv_type]),
            }.items()
        ):
            _, ax = plot_grouped_boxes(
                data={
                    group: {
                        reg: np.concatenate(metrics[group][reg])
                        for reg in encoder_mb.regions
                    }
                    for group in regressors_
                },
                colors=colors_region,
                plot_points=plot_points,
                ax=axes[i][j],
            )
            ax.set_ylim([-0.15, 0.2])
            ax.legend(loc="upper right")
            ax.set_title(strategy)

    fig.suptitle(rf"{subj_id}, $\Delta r^2$")
    save_fig(
        fig,
        FIGURES_DIR / "cv_d_r2" / "boxplot_comp" / "grouped" / "no_svd",
        f"{subj_id}_dr2_comp-{points_str}.png",
    )

## plot scatters

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from core.viz import plot_scatter
from utils.colors import colors_region
from utils.paths import VARS_DIR, FIGURES_DIR
from utils.viz_utils import save_fig


fpath = VARS_DIR / "cv_d_r2" / "grouped" / "no_svd"
with open(fpath / f"{subj_id}_cvr2s.pkl", "rb") as f:
    cvr2s = pickle.load(f)
with open(fpath / f"{subj_id}_dr2s.pkl", "rb") as f:
    dr2s = pickle.load(f)

n_regions = len(encoder_mb.regions)
fig, axes = plt.subplots(
    ncols=n_regions, nrows=2, figsize=(2.5 * n_regions, 5), tight_layout=True
)

for i, (metric, (metrics, lim)) in enumerate(
    {r"cv $r^2$": (cvr2s, [-0.25, 0.8]), r"$\Delta r^2$": (dr2s, [-0.3, 0.3])}.items()
):
    for j, reg in enumerate(encoder_mb.regions):
        plot_scatter(
            np.concatenate(metrics["mb"]["spe"][reg]),
            np.concatenate(metrics["mf"]["rpe"][reg]),
            xlabel=f"spe, {metric}",
            ylabel=f"rpe, {metric}",
            color=colors_region[reg],
            title=reg,
            mn=lim[0],
            mx=lim[1],
            add_unity=True,
            add_lr=True,
            ax=axes[i][j],
        )
fig.suptitle(subj_id)

save_fig(
    fig, FIGURES_DIR / "cv_d_r2" / "spe_v_rpe" / "no_svd", f"{subj_id}_spe_v_rpe.png"
)